In [1]:
!pip install pyspark

In [2]:
import socket
import time
import threading

In [3]:
messages = [
    "spark streaming dstream",
    "spark spark streaming",
    "big data spark",
    "real time big data",
    "apache spark streaming"
]

with open("messages.txt", "w") as f:
    for msg in messages:
        f.write(msg + "\n")

In [4]:
import random

def start_socket_server():
    host = "localhost"
    port = 9999

    s = socket.socket()
    s.bind((host, port))
    s.listen(1)

    print("Socket server started on port 9999...")
    conn, addr = s.accept()
    print("Client connected:", addr)

    with open("messages.txt", "r") as f:
        lines = f.readlines()

    while True:
        msg = random.choice(lines).strip()
        conn.send((msg + "\n").encode())
        time.sleep(2)

threading.Thread(target=start_socket_server, daemon=True).start()

In [5]:
from pyspark import SparkContext
from pyspark.streaming import StreamingContext

sc = SparkContext.getOrCreate()

ssc = StreamingContext(sc, 5)

/usr/local/lib/python3.12/dist-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


In [6]:
lines = ssc.socketTextStream("localhost", 9999)

In [7]:
words = lines.flatMap(lambda line: line.split(" "))

pairs = words.map(lambda word: (word, 1))

counts = pairs.reduceByKey(lambda a, b: a + b)

In [8]:
counts.pprint()

In [9]:
ssc.start()

ssc.awaitTerminationOrTimeout(30)

ssc.stop(stopSparkContext=False)

Client connected: ('127.0.0.1', 47846)
-------------------------------------------
Time: 2026-01-21 22:17:05
-------------------------------------------

-------------------------------------------
Time: 2026-01-21 22:17:10
-------------------------------------------
('apache', 1)
('streaming', 1)
('big', 1)
('spark', 1)
('real', 1)
('time', 1)
('data', 1)

-------------------------------------------
Time: 2026-01-21 22:17:15
-------------------------------------------
('streaming', 3)
('dstream', 1)
('spark', 5)

-------------------------------------------
Time: 2026-01-21 22:17:20
-------------------------------------------
('big', 1)
('apache', 1)
('streaming', 1)
('data', 1)
('spark', 2)

-------------------------------------------
Time: 2026-01-21 22:17:25
-------------------------------------------
('apache', 1)
('streaming', 2)
('big', 1)
('spark', 3)
('real', 1)
('time', 1)
('data', 1)

-------------------------------------------
Time: 2026-01-21 22:17:30
----------------------

4. Qu’est-ce qu’un micro-batch dans DStream ?

Un micro-batch est une petite portion de données collectée pendant un intervalle de temps fixe (ex: 5 secondes).
Chaque micro-batch est traité comme un RDD par Spark Streaming.

5. Sur quelle structure repose un DStream ?

Un DStream repose sur une suite de RDDs (Resilient Distributed Datasets), chacun représentant un micro-batch de données.

6. Quelle est la durée du batch utilisée ?

La durée du batch utilisée est 5 secondes, définie ici :

ssc = StreamingContext(sc, 5)